In [1]:
import numpy as np
import pandas as pd

class KNN:
    def __init__(self, k=5):
        self.k = k
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y

    def predict_L2(self, X_test, k):
        diff = X_test[:, np.newaxis, :] - self.X_train[np.newaxis, :, :]
        distances = np.sqrt(np.sum(diff ** 2, axis=2))

        nearest_indices = np.argsort(distances, axis=1)[:, :k]
        nearest_labels = self.y_train[nearest_indices]

        y_pred = np.where(np.sum(nearest_labels, axis=1) >= 0, 1, -1)
        return y_pred

    def predict_L1(self, X_test, k):
        diff = X_test[:, np.newaxis, :] - self.X_train[np.newaxis, :, :]
        distances = np.sum(np.abs(diff), axis=2)

        nearest_indices = np.argsort(distances, axis=1)[:, :k]
        nearest_labels = self.y_train[nearest_indices]

        y_pred = np.where(np.sum(nearest_labels, axis=1) >= 0, 1, -1)
        return y_pred

def compute_accuracy(y_true, y_pred):
    return float(np.mean(y_true == y_pred))

def standardize(X_train, X_test):
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)
    std[std == 0] = 1.0

    X_train_std = (X_train - mean) / std
    X_test_std = (X_test - mean) / std
    return X_train_std, X_test_std

def get_pearson_indices(X, y, m):
    mean_X = np.mean(X, axis=0)
    mean_y = np.mean(y)

    numerator = np.sum((X - mean_X) * (y - mean_y).reshape(-1, 1), axis=0)
    denom_X = np.sum((X - mean_X) ** 2, axis=0)
    denom_y = np.sum((y - mean_y) ** 2)

    denominator = np.sqrt(denom_X * denom_y)
    denominator[denominator == 0] = 1e-8

    r = np.abs(numerator / denominator)
    indices = np.argsort(r)[::-1][:m]
    return indices

if __name__ == "__main__":
    np.random.seed(42)
    X_tr = np.random.randn(100, 10)
    y_tr = np.random.choice([-1, 1], size=100)
    X_te = np.random.randn(20, 10)
    y_te = np.random.choice([-1, 1], size=20)

    knn = KNN()
    knn.fit(X_tr, y_tr)

    print("Task A (L2 Distances):")
    for k in [1, 2, 5, 10, 100]:
        preds = knn.predict_L2(X_te, k)
        print(f"k={k}, Accuracy={compute_accuracy(y_te, preds):.2f}")

    print("\nTask B (Standardized + Pearson):")
    X_tr_std, X_te_std = standardize(X_tr, X_te)
    knn_std = KNN()
    for m in [2, 5, 8]:
        top_m_indices = get_pearson_indices(X_tr_std, y_tr, m)
        knn_std.fit(X_tr_std[:, top_m_indices], y_tr)
        preds = knn_std.predict_L2(X_te_std[:, top_m_indices], 20)
        print(f"m={m}, k=20, Accuracy={compute_accuracy(y_te, preds):.2f}")

    print("\nTask C (Standardized + L1):")
    knn_std.fit(X_tr_std, y_tr)
    preds = knn_std.predict_L1(X_te_std, 20)
    print(f"All features, k=20, L1 Accuracy={compute_accuracy(y_te, preds):.2f}")

Task A (L2 Distances):
k=1, Accuracy=0.35
k=2, Accuracy=0.45
k=5, Accuracy=0.40
k=10, Accuracy=0.45
k=100, Accuracy=0.45

Task B (Standardized + Pearson):
m=2, k=20, Accuracy=0.40
m=5, k=20, Accuracy=0.30
m=8, k=20, Accuracy=0.70

Task C (Standardized + L1):
All features, k=20, L1 Accuracy=0.35
